# Marco 2 — Primeira chain de propostas com fontes

Execute com requirements-notebook.txt e requirements-ia.txt instalados no mesmo kernel.
O notebook é offline por padrão: usa respostas manuais identificadas, não comprova compreensão de LLM.
A opção OpenAI no terminal usa a mesma chain com gpt-4o-mini, sem enviar o backup real.
Propostas ficam aguardando revisão; nenhuma agenda, pagamento ou projeto operacional é gravado.


In [1]:
from pathlib import Path
import json, sys, tempfile

# CAMINHOS PORTÁTEIS: localiza o projeto no diretório atual ou em seus pais, sem caminho pessoal.
raiz = None
for base in (Path.cwd(), *Path.cwd().parents):
    for candidato in (base, base / "Agentes Cognitivos com LangChain" / "Projeto final 2"):
        if (candidato / "PLANO_PROJETO_SECRETARIA_IA.md").is_file() and (candidato / "src/secretaria").is_dir():
            raiz = candidato
            break
    if raiz:
        break
if raiz is None:
    raise RuntimeError("Abra o notebook com a pasta completa do projeto e seu ambiente Python.")
sys.path.insert(0, str(raiz / "src"))
from secretaria.importador import ler_arquivo, interpretar
from secretaria.banco import Banco, Contexto
print("Python:", sys.version.split()[0])

from secretaria.chain_extracao import criar_chain
from secretaria.extracao import preparar_lotes, analisar_snapshot
from secretaria.fixtures_extracao import criar_modelo_fixture
from secretaria.apresentacao import exibir_analise


Python: 3.12.6


## 1. Snapshot completo e lotes com vizinhos

SQLite fornece todos os registros autorizados. Lotes de oito mensagens têm uma mensagem vizinha de cada lado.
É cobertura do processamento, não afirmação de que o modelo entendeu todas as mensagens.


In [2]:
# SQLITE/CONTEXTO: corpus fictício e identidade fornecida pela aplicação.
ambiente_demo = tempfile.TemporaryDirectory(prefix="secretaria_chain_notebook_")
banco_arquivo = Path(ambiente_demo.name) / "demo.sqlite3"
ctx = Contexto("demo", "ana")
with Banco(banco_arquivo) as banco:
    banco.preparar_demo()
    corpus = banco.selecionar_corpus(ctx, "Demonstração fictícia")
    banco.importar(ctx, corpus, raiz / "dados/ficticios/conversa_empresa.txt")
    snapshot = banco.snapshot_analise(ctx, corpus)
lotes = preparar_lotes(snapshot)
assert len(snapshot["mensagens"]) == 40 and len(lotes) == 5
assert [m["id"] for l in lotes for m in l["mensagens_foco"]] == [m["id"] for m in snapshot["mensagens"]]
print("40 mensagens em 5 lotes, sem truncamento.")


40 mensagens em 5 lotes, sem truncamento.


## 2. ChatPromptTemplate, modelo e parser

LCEL liga prompt | modelo | PydanticOutputParser. O decorador @chain demonstra a mesma sequência imperativa.
O parser prova estrutura; os IDs/trechos/valores ainda passam pela validação de fontes no serviço.


In [3]:
# RUNNABLE/INVOKE: o modelo injetado aqui é fixture manual, não OpenAI.
entrada = {"pacote": json.dumps(lotes[0], ensure_ascii=False)}
lcel = criar_chain(criar_modelo_fixture(), "lcel")
imperativa = criar_chain(criar_modelo_fixture(), "imperativa")
a = lcel.invoke(entrada)
b = imperativa.invoke(entrada)
assert a.model_dump() == b.model_dump()
print("LCEL e @chain produziram o mesmo contrato na fixture.")


LCEL e @chain produziram o mesmo contrato na fixture.


## 3. Batch e stream de objetos completos

batch executa as leituras independentes. Nesta implementação imperativa, stream emite um objeto completo;
não demonstra streaming de tokens nem exibe propostas parciais como fatos verificados.


In [4]:
# RUNNABLE/BATCH/STREAM: demonstra métodos reais, com respostas roteirizadas identificadas.
respostas = lcel.batch([{"pacote": json.dumps(l, ensure_ascii=False)} for l in lotes], config={"max_concurrency": 2})
partes = list(imperativa.stream(entrada))
assert len(respostas) == 5 and len(partes) == 1
print("batch:", len(respostas), "lotes; stream imperativo:", len(partes), "objeto completo.")


batch: 5 lotes; stream imperativo: 1 objeto completo.


## 4. Propostas, fontes e cobertura

Condição de entrada é diferente de pagamento informado. Visita confirmada na conversa é diferente de registro
gravado na agenda. Algumas interpretações ficam pendentes; não escolhemos silenciosamente a última versão.


In [5]:
# VALIDAÇÃO: o serviço confere fontes, valores escritos e cobertura antes da apresentação.
resultado = analisar_snapshot(snapshot, criar_modelo_fixture(), "fixture_manual")
assert resultado["status"] == "completa"
assert resultado["mensagens_em_lotes_concluidos"] == 40
assert len(resultado["propostas"]) == 8
assert all(p["status"] == "aguardando_revisao" for p in resultado["propostas"])
assert resultado["gravacoes_operacionais"] == 0
exibir_analise(resultado)
with Banco(banco_arquivo) as banco:
    assert banco.contar(ctx, corpus) == 40
    assert banco.con.execute("SELECT COUNT(*) FROM importacoes").fetchone()[0] == 1



PROPOSTAS PARA REVISÃO — nenhum registro operacional foi gravado
Origem: fixture_manual | Composição: lcel
Cobertura dos lotes: 40 / 40
[FIXTURE LOCAL] Respostas manuais; nenhuma chamada de LLM.

1. PROJETO | Alfa | relato
Implantação do portal; responsável informado Ana.
Fonte M0003: Projeto Alfa: implantação do portal de atendimento.
Cliente: Alfa Serviços.
Responsável: Ana.

2. FINANCEIRO | Alfa | previsto
Orçamento Alfa informado em R$ 10.000,00.
Tipo: orcamento_proposto
Valor: R$ 10.000,00
Fonte M0004: Orçamento Alfa: R$ 10.000,00. Entrada combinada de 30%; restante em 20/11/2026.

3. FINANCEIRO | Alfa | previsto
Entrada combinada de 30%; não comprova pagamento.
Tipo: condicao_prevista
Percentual informado: 30%
Fonte M0004: Orçamento Alfa: R$ 10.000,00. Entrada combinada de 30%; restante em 20/11/2026.

4. AGENDA | Alfa | confirmacao
Visita Alfa confirmada na conversa; local ainda pendente.
Início: 2026-11-10T14:00:00-03:00 | Fim: 2026-11-10T15:00:00-03:00
Fonte M0008: Confirmo v

## 5. Contrato inválido vira falha explícita

A análise não passa a ser completa se um lote falhar. Não há reparo infinito nem resposta inventada como fallback.


In [6]:
from langchain_core.messages import AIMessage
from langchain_core.runnables import RunnableLambda
# PARSER/ERROS: caso adverso local, não uma falha observada de API real.
invalido = RunnableLambda(lambda _: AIMessage(content="não é JSON"))
pequeno = {**snapshot, "mensagens": snapshot["mensagens"][:1]}
falha = analisar_snapshot(pequeno, invalido, "fixture_adversa")
assert falha["status"] == "parcial" and falha["propostas"] == []
print("Falha identificada:", falha["lotes"][0]["tipo_erro"])


Falha identificada: OutputParserException


## Limites e próximo passo

Isto é uma chain de interpretação, não um agente com ferramentas. Oito propostas são respostas manuais
do modo fixture, não gabarito de todas as informações possíveis. Fontes e valores consistentes não provam
toda a interpretação semântica; revisão humana e verificador completo vêm em incrementos posteriores.
Múltiplos lotes podem gerar propostas semanticamente repetidas. Não foi implementada consolidação final
nem aplicação de confirmação. O corpus real privado não é selecionado para envio externo neste fluxo.

Para chamada real: opção 5 no terminal ou --analisar api --arquivo-env caminho/.env.
Não coloque chave no notebook nem em arquivos versionados.
